# Exploratory Data Analysis(EDA) using SQL

## Objective

In this section, I performed Exploratory Data Analysis (EDA) using SQL queries to understand patterns and relationships in the dataset.

The goal of this analysis is to identify key factors that influence the success of Falcon 9 landings.

In [1]:
!pip install sqlalchemy==1.3.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 141.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... one
  Created wheel for sqlalchemy: filename=SQLAlchemy-1.3.9-cp312-cp312-linux_x86_64.whl size=1160111 sha256=fecc963673967817824100103286d0c88da0433e94c1e48d99d94485749461aa
  Stored in directory: /home/jupyterlab/.cache/pip/wheels/b3/1c/42/0e26b8d512adc6bce10ff71a05229366b4ccec641cd3b42111
Successfully built sqlalchemy
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.37
    Uninstalling SQLAlchemy-2.0.37:
      Successfully uninstalled SQLAlchemy-2.0.37
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyterhub 5.2.1 requires SQLAlchemy>=1.4.1, but you have sqlalchemy 1.3.9 which is incompatible.


#### I am first loading  the SQL extension and establishing  a connection with the database

In [2]:
!pip install ipython-sql
!pip install ipython-sql prettytable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 144.2 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.3.9
    Uninstalling SQLAlchemy-1.3.9:
      Successfully uninstalled SQLAlchemy-1.3.9


In [3]:
%load_ext sql

In [4]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [5]:
!pip install -q pandas

In [6]:
%sql sqlite:///my_data1.db

In [7]:
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

The below code is added to remove blank rows from table

In [8]:
#DROP THE TABLE IF EXISTS

%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


[]

In [9]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.


[]

In [10]:
%%sql
SELECT DISTINCT "LaunchSite"
FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


"""LaunchSite"""
LaunchSite


In [11]:
%%sql
SELECT*
FROM SPACEXTABLE
WHERE "LaunchSite" LIKE 'CCA%';


 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome


In [12]:
%%sql
SELECT SUM("PayloadMass")
FROM SPACEXTABLE
WHERE "CUSTOMER" = 'NASA(CRS)';

 * sqlite:///my_data1.db
Done.


"SUM(""PayloadMass"")"
None


In [13]:
%%sql
SELECT AVG("PayloadMass")
FROM SPACEXTABLE
WHERE "BoosterVersion" = 'F9 V1.1';

 * sqlite:///my_data1.db
Done.


"AVG(""PayloadMass"")"
None


In [14]:
%%sql
SELECT MIN("Date")
FROM SPACEXTABLE
WHERE "Landing_Outcomes"= 'SUCCESS(ground_pad)';


 * sqlite:///my_data1.db
Done.


"MIN(""Date"")"
None


In [15]:
%%sql
SELECT "Booster Version"
FROM SPACEXTABLE
WHERE "Landing_Outcomes" = 'Success( drone ship)'
AND "PayloadMass" BETWEEN 4000 AND 6000;

 * sqlite:///my_data1.db
Done.


"""Booster Version"""


Total number of successful and failure mission outcomes


In [16]:
%%sql
SELECT "Outcome", COUNT(*)
FROM SPACEXTABLE
GROUP BY "Outcome"

 * sqlite:///my_data1.db
Done.


"""Outcome""",COUNT(*)
Outcome,101


In [17]:
%%sql
SELECT "Booster Version "
FROM SPACEXTABLE
WHERE "PayloadMass" = (
    SELECT MAX("PayloadMass")
    FROM SPACEXTABLE
);


 * sqlite:///my_data1.db
Done.


"""Booster Version """
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version
Booster Version


In [18]:
%%sql
SELECT 
    substr("Date",6,2) AS Month,
    "Landing Outcome",
    "BoosterVersion",
    "LaunchSite"
FROM SPACEXTABLE
WHERE substr("Date",1,4) = '2015'
AND "Landing_Outcome" LIKE 'Failure%'
AND "Landing_Outcome" LIKE '%droneship%';

 * sqlite:///my_data1.db
Done.


Month,"""Landing Outcome""","""BoosterVersion""","""LaunchSite"""


### Ranking the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [19]:
%%sql
SELECT "Landing_Outcome", COUNT(*) AS Count
FROM SPACEXTABLE
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome"
ORDER BY Count DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
